# 🥉 Notebook 02: Ingestion from Raw CSV to Bronze Delta Layer

## 🎯 Objectives & "Why We Do This" Proofs
1. **Explicit Schema Enforcement vs `inferSchema`**: Demonstrate why specifying `StructType` is 3x-5x faster and prevents schema corruption.
2. **Bronze Audit Metadata**: Append lineage tracking columns (`_ingested_at`, `_source_file`, `_batch_id`) to maintain complete data provenance.
3. **Delta Lake Storage**: Write landing data into Delta format to enable ACID transactions and schema evolution.

---

In [0]:
# Databricks notebook source
import time
import uuid
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType, DateType

LANDING_PATH = "/tmp/mini_project2/landing"
BRONZE_PATH = "/tmp/mini_project2/bronze"
BATCH_ID = str(uuid.uuid4())[:8]

print(f"Reading Raw CSVs from: {LANDING_PATH}")
print(f"Writing Bronze Delta Tables to: {BRONZE_PATH}")
print(f"Ingestion Batch ID: {BATCH_ID}")

### 💡 PROOF 1: Why Explicit Schema (`StructType`) instead of `inferSchema=True`?

In [0]:
# Benchmark A: Reading CSV with inferSchema=True (Reads CSV twice, slow, risky for production)
t0 = time.time()
df_infer = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{LANDING_PATH}/orders")
count_infer = df_infer.count()
t1 = time.time()
infer_time = round(t1 - t0, 3)

# Explicit Schema Definition
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("status", StringType(), True)
])

# Benchmark B: Reading CSV with Explicit Schema (Reads CSV once, fast, production-safe)
t0 = time.time()
df_explicit = spark.read.option("header", "true").schema(orders_schema).csv(f"{LANDING_PATH}/orders")
count_explicit = df_explicit.count()
t1 = time.time()
explicit_time = round(t1 - t0, 3)

print("=== 📊 DEMONSTRATION: inferSchema vs Explicit Schema ===")
print(f"⏱️ inferSchema=True Duration: {infer_time} seconds")
print(f"⏱️ Explicit Schema Duration : {explicit_time} seconds")
print(f"💡 WHY WE USE EXPLICIT SCHEMA: inferSchema scans the entire file TWICE to guess types, slowing down ingestion and risking unexpected schema shifts when columns change!")

### 💡 PROOF 2: Why Append Bronze Audit Columns (`_ingested_at`, `_source_file`, `_batch_id`)?

In [0]:
# Define remaining schemas
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("region", StringType(), True),
    StructField("tier", StringType(), True),
    StructField("signup_date", StringType(), True)
])

txns_schema = StructType([
    StructField("txn_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("txn_amount", StringType(), True),
    StructField("txn_timestamp", StringType(), True),
    StructField("txn_status", StringType(), True)
])

# Ingest & Enrich Orders with Audit Lineage Columns
df_bronze_orders = df_explicit \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_batch_id", F.lit(BATCH_ID))

df_bronze_orders.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/orders")

print("=== 📊 DEMONSTRATION: Bronze Lineage Audit Columns ===")
display(df_bronze_orders.select("order_id", "customer_id", "_ingested_at", "_source_file", "_batch_id").limit(3))
print("💡 WHY WE USE AUDIT COLUMNS: Bronze layer must preserve raw raw state + audit metadata so engineers can trace data lineage back to exact source file and ingestion timestamp during debugging.")

In [0]:
# Ingest Customers & Transactions to Bronze Delta
df_bronze_cust = spark.read.option("header", "true").schema(customers_schema).csv(f"{LANDING_PATH}/customers") \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_batch_id", F.lit(BATCH_ID))
df_bronze_cust.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/customers")

df_bronze_txns = spark.read.option("header", "true").schema(txns_schema).csv(f"{LANDING_PATH}/transactions") \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_batch_id", F.lit(BATCH_ID))
df_bronze_txns.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/transactions")

spark.sql(f"CREATE TABLE IF NOT EXISTS bronze_orders USING DELTA LOCATION '{BRONZE_PATH}/orders'")
spark.sql(f"CREATE TABLE IF NOT EXISTS bronze_customers USING DELTA LOCATION '{BRONZE_PATH}/customers'")
spark.sql(f"CREATE TABLE IF NOT EXISTS bronze_transactions USING DELTA LOCATION '{BRONZE_PATH}/transactions'")
print("✅ Bronze Delta tables registered in Metastore.")

In [0]:
%sql
-- Inspect Bronze Delta History
DESCRIBE HISTORY bronze_orders;